# 07 - LLM-Enhanced Recommendation System

This notebook uses `student_recommendation_features.csv` as the direct input for the LLM recommendation layer.

The design is thesis-safe: rule-based recommendations provide the factual content, and the LLM only rewrites the output into a clear personalized report.

## 1. Setup

In [1]:
import os
import json
from pathlib import Path

import pandas as pd
import numpy as np

from blended_learning.config.settings import settings


pd.set_option("display.max_colwidth", 250)
pd.set_option("display.max_columns", 100)

path = settings.root / "data" / "processed" / "student_recommendation_features.csv"
print("Using data file:", path)

Using data file: C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\data\processed\student_recommendation_features.csv


In [2]:
# from blended_learning.llm.openrouter import OpenRouterStudentRecommender

# recommender = OpenRouterStudentRecommender(
#     model="openai/gpt-oss-120b:free"
# )

# package = {
#     "student_id": "S001",
#     "student_segment": 1,
#     "student_segment_label": "Needs structured support",
#     "cluster_label": "Cluster 1",

#     "open_strengths_clean": "I can learn well when the teacher explains clearly.",
#     "open_challenges_clean": "I have difficulty managing online learning tasks.",

#     "strength_sentiment_label": "positive",
#     "challenge_sentiment_label": "negative",
#     "strength_compound": 0.45,
#     "challenge_compound": -0.40,

#     "strength_themes": ["teacher explanation", "guided learning"],
#     "challenge_themes": ["online task management", "self-study difficulty"],

#     "recommendation_tags": ["structured_schedule", "teacher_guidance"],
#     "segment_default_tags": ["guided_support"],
#     "final_recommendation_tags": ["structu red_schedule", "teacher_guidance", "guided_support"],

#     "rule_based_recommendations": [
#         {
#             "title": "Follow a weekly study schedule",
#             "recommendation": "Use a clear weekly plan to complete online tasks and review materials step by step."
#         },
#         {
#             "title": "Ask for teacher guidance",
#             "recommendation": "Prepare questions before class and ask the teacher for clarification when online materials are difficult."
#         }
#     ]
# }

# report = recommender.generate_report(package)

# print("LLM recommendation report:")
# print(report)

In [3]:
from blended_learning.llm.openrouter import OpenRouterStudentRecommender

recommender = OpenRouterStudentRecommender(
    model="openai/gpt-oss-120b:free"
)

input_csv = settings.root / "data" / "processed" / "student_recommendation_features.csv"
output_csv = settings.root / "data" / "processed" / "student_recommendation_reports.csv"


output_df = recommender.generate_reports_from_csv(
    input_csv=input_csv,
    output_csv=output_csv,
    limit=None,
    resume=True,
    save_each_student=True
)

[1/567] Skipping already processed student: e20210686
[2/567] Skipping already processed student: e20241146
[3/567] Skipping already processed student: e20240609
[4/567] Skipping already processed student: e20240542
[5/567] Skipping already processed student: e20220287
[6/567] Skipping already processed student: e20210180
[7/567] Skipping already processed student: e20241245
[8/567] Skipping already processed student: e20221090
[9/567] Skipping already processed student: e20250089
[10/567] Skipping already processed student: e20240950
[11/567] Skipping already processed student: e20240099
[12/567] Skipping already processed student: e20241378
[13/567] Skipping already processed student: e20210635
[14/567] Skipping already processed student: e20240052
[15/567] Skipping already processed student: e20240937
[16/567] Skipping already processed student: e20250073
[17/567] Skipping already processed student: e20240164
[18/567] Skipping already processed student: e20250120
[19/567] Skipping a

**Student Learning Profile**  
- **Student ID:** e20210528  
- **Segment:** Cluster 2 – *Highly Engaged (Active) Learners*  

Because the open‑ended responses about strengths and challenges are empty, the recommendations below are drawn primarily from the characteristics of the “Highly Engaged (Active) Learners” segment and the rule‑based tags assigned to this student.

---

### Main Strengths
*No specific strengths were reported, but as a Highly Engaged learner you are typically motivated, participative, and comfortable with active learning formats.*

---

### Main Challenges
*No specific challenges were reported. The segment profile suggests you may benefit from structured support that matches your active learning style.*

---

### Personalized Recommendations  

| Focus | What to Do | Why it Helps |
|-------|------------|--------------|
| **Content Access** | Upload or request early access to slides, videos, reference links, exercises, and any other course materials. | Having the materials ahead of time lets you prepare, review, and stay ahead of class activities. |
| **Digital Skills** | Take short tutorials on the LMS, online research tools, and any educational technologies used in the course. | Strong digital skills let you navigate resources quickly and make the most of online learning tools. |
| **Engagement** | Participate in quizzes, games, practical tasks, discussion boards, and other interactive activities. | Interactive elements keep your high energy focused and deepen understanding. |
| **Self‑Paced Learning** | Use recorded lessons and weekly learning paths; review materials at your own speed when needed. | Even active learners benefit from the flexibility to revisit complex topics or accelerate through familiar ones. |

---

### Short Action Plan (Next 2 Weeks)

1. **Day 1‑2:** Check the course portal for any missing slides or videos; request early uploads from the instructor if needed.  
2. **Day 3‑5:** Complete a 30‑minute tutorial on the LMS’s key features (e.g., submitting assignments, using discussion forums).  
3. **Day 6‑9:** Join at least one online quiz or game related to the current module; post a comment or question in the discussion board.  
4. **Day 10‑12:** Review the recorded lecture for the upcoming topic and create a personal “weekly learning path” checklist.  
5. **Day 13‑14:** Reflect on which resources helped you most and note any gaps to discuss with the instructor next week.  

By following these steps, you’ll reinforce the strengths of an active learner while ensuring you have the resources, skills, and flexibility needed for success in a blended learning environment.

In [4]:
import pandas as pd
import json
from pathlib import Path
from blended_learning.config.settings import settings

# Input CSV file
csv_path = settings.root / "data" / "processed" / "student_recommendation_reports.csv"

# Output files
out_ipynb_path = settings.root / "notebook" / "student_recommendation_reports_markdown.ipynb"
out_md_path = settings.root / "notebook" / "student_recommendation_reports_markdown.md"

# Create output folder if it does not exist
out_ipynb_path.parent.mkdir(parents=True, exist_ok=True)

# Load CSV
df = pd.read_csv(csv_path)

required_columns = [
    "student_id",
    "student_segment_label",
    "final_recommendation_tags",
    "llm_recommendation_report"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")


def make_student_markdown(row):
    student_id = str(row["student_id"])
    segment = str(row["student_segment_label"])
    tags = str(row["final_recommendation_tags"])
    report = "" if pd.isna(row["llm_recommendation_report"]) else str(row["llm_recommendation_report"])

    markdown_text = f"""## Student ID: `{student_id}`

**Segment:** {segment}

**Recommendation Tags:** `{tags}`

---

{report}
"""

    return markdown_text


intro_markdown = f"""# Student Recommendation Reports

This notebook was generated from:

`{csv_path}`

**Total student records:** {len(df)}

Each section below contains:

- Student ID
- Student segment label
- Final recommendation tags
- LLM-generated recommendation report
"""

cells = [
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": intro_markdown.splitlines(keepends=True)
    }
]

all_markdown_parts = [intro_markdown, "\n\n---\n"]

for _, row in df.iterrows():
    student_markdown = make_student_markdown(row)

    cells.append(
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": student_markdown.splitlines(keepends=True)
        }
    )

    all_markdown_parts.append(student_markdown)
    all_markdown_parts.append("\n\n---\n")


notebook = {
    "cells": cells,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.x"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

# Save .ipynb notebook
with open(out_ipynb_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=2)

# Optional: also save a normal Markdown file
with open(out_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(all_markdown_parts))

print("Conversion complete.")
print(f"Notebook saved to: {out_ipynb_path}")
print(f"Markdown saved to: {out_md_path}")
print(f"Rows converted: {len(df)}")

Conversion complete.
Notebook saved to: C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\notebook\student_recommendation_reports_markdown.ipynb
Markdown saved to: C:\Users\Tepy\Documents\tepy\Final Internship Docs\blended_learning\notebook\student_recommendation_reports_markdown.md
Rows converted: 606
